# Stacking with logistic regression on Exp1, Exp3 and Exp5

Our best submission (public 0.98047) is the equal-weight mean of three models' probabilities: Exp1 (ConvNeXt-base), Exp3 (EfficientNetV2-S) and Exp5 (BiT ResNetV2-50x1). As the assignment suggested stacking with logistic regression we introduce a second-level model that learns how to combine the three members, instead of giving each the same weight.

**What the meta-model may learn from.** A stacking model must be fitted on member predictions for images the members did not train on. Otherwise it learns how confident the members are on their own training images, which is close to 1 everywhere. All three members were trained on folds 1–4, so the only such images are the 2,452 photos of fold 0. That is the whole training set of the meta-model, with 8–10 photos per named pet.

**What stays fixed.** The members, their saved fold-0 and test probabilities, the label order and the 0.25 threshold. Nothing is retrained and no image is predicted again. The notebook only reads saved files. The submission CSV and run record are written only with `BUILD_SUBMISSION=True`. Kaggle upload is always manual.

In [1]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

BUILD_SUBMISSION = os.environ.get('STAGE2_BUILD_SUBMISSION', '0') == '1'
SUBMIT_TO_KAGGLE = False  # No submission API call exists in this notebook.

RUN_ID = 'stacking_logreg_exp1_exp3_exp5_fold0'
VALIDATION_FOLD = 0
THRESHOLD = 0.25
C = 1.0  # scikit-learn's default L2 strength; the other values below are a sensitivity check, not a search.
C_GRID = (0.01, 0.1, 1.0, 10.0)
CLIP = 1e-6  # Probabilities are clipped to [1e-6, 1 - 1e-6] before the logit, so features stay within +-13.8.
INNER_FOLDS = 5
REPEATS = 10
SEED = 42
BEST_SUBMISSION = 'submissions/exp4_ensemble_exp1_exp3_exp5_mean_025.csv'  # Public 0.98047.
MEMBERS = {
    'exp1': 'convnext_standard_fold0_seed42_20261018_full1',
    'exp3': 'efficientnetv2s_fold0_seed42',
    'exp5': 'bit_r50x1_fold0_seed42',
}
print('build submission:', BUILD_SUBMISSION)

build submission: False


## Load the members' fold-0 and test predictions

As in notebook 13, each member must be a completed fold-0 run on the same fold file and label order, its re-scored fold-0 macro F1 must match its record, and its saved test probabilities must come from the same checkpoint. The arrays have the shape (member, image, label).

In [2]:
start = Path(os.environ.get('STAGE2_ROOT', Path.cwd())).resolve()
ROOT = next((path for path in (start, *start.parents) if (path / 'folds.csv').is_file() and (path / 'stage2' / 'experiment.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Launch Jupyter from this repository or a subdirectory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from stage2.ensemble import mean_probabilities, read_probabilities
from stage2.experiment import LABEL_COLUMNS, form_submission, read_csv, score_probabilities, split_rows, write_probabilities, write_run_record, write_submission
from stage2.run_records import file_sha256

DATA_DIR = ROOT / 'data'
_, validation_rows = split_rows(DATA_DIR, ROOT / 'folds.csv', VALIDATION_FOLD)
validation_ids = [row['image_id'] for row in validation_rows]
validation_targets = np.asarray([[int(row[label]) for label in LABEL_COLUMNS] for row in validation_rows])
test_ids = [row['image_id'] for row in read_csv(DATA_DIR / 'test.csv', ('image_id',))]
fold_sha256 = file_sha256(ROOT / 'folds.csv')

records, validation_members, test_members = {}, [], []
for name, run_id in MEMBERS.items():
    run_dir = ROOT / 'experiments' / name / 'results' / run_id
    record = json.loads((run_dir / f'{run_id}.json').read_text())
    if record['status'] != 'completed' or record['fold'] != VALIDATION_FOLD:
        raise ValueError(f'{run_id} is not a completed fold-{VALIDATION_FOLD} run.')
    if record['data']['fold_sha256'] != fold_sha256 or record['data']['label_columns'] != list(LABEL_COLUMNS):
        raise ValueError(f'{run_id} used a different fold file or label order.')
    probabilities = read_probabilities(run_dir / f'{run_id}_val.csv', validation_ids)
    rescored = score_probabilities(validation_targets, probabilities, THRESHOLD)['macro_f1']
    if not np.isclose(rescored, record['metrics']['macro_f1'], rtol=0, atol=1e-9):
        raise ValueError(f'{run_id}: re-scored macro F1 {rescored} differs from its record.')
    test_dir = ROOT / 'experiments' / name / 'results' / f'{run_id}_test'
    if json.loads((test_dir / f'{run_id}_test.json').read_text())['source_checkpoint_sha256'] != record['artifacts']['best_checkpoint_sha256']:
        raise ValueError(f'{run_id}: the saved test probabilities come from another checkpoint.')
    records[name] = record
    validation_members.append(probabilities)
    test_members.append(read_probabilities(test_dir / f'{run_id}_test.csv', test_ids))
    print(f'{name}: {run_id} | fold-0 macro F1 {rescored:.5f} (matches record) | test probabilities loaded')
validation_members, test_members = np.stack(validation_members), np.stack(test_members)
print('fold-0 positives per label:', dict(zip(LABEL_COLUMNS, validation_targets.sum(axis=0).tolist())))

exp1: convnext_standard_fold0_seed42_20261018_full1 | fold-0 macro F1 0.97518 (matches record) | test probabilities loaded
exp3: efficientnetv2s_fold0_seed42 | fold-0 macro F1 0.99831 (matches record) | test probabilities loaded
exp5: bit_r50x1_fold0_seed42 | fold-0 macro F1 0.99851 (matches record) | test probabilities loaded
fold-0 positives per label: {'Cat': 1015, 'Dog': 1004, 'Moris': 10, 'Motya': 9, 'Biatrix': 8}


## The meta-model

**One logistic regression per label, on that label's three member logits.** For Moris, for example, the inputs are the Moris logits of Exp1, Exp3 and Exp5, and the model learns three weights and an intercept. We use logits rather than probabilities because the members' probabilities are mostly within 0.001 of 0 or 1. On the probability scale, 0.999 and 0.999999 look the same, while their logits (6.9 and 13.8) do not.

**Why so few inputs.** Each named pet has only 8–10 positives in fold 0. Four parameters per label are already plenty for that. A variant that gives every label all 15 member logits (so that, say, a confident Moris could raise Cat) is evaluated below for comparison.

**Regularization.** L2 with scikit-learn's default strength `C = 1`. With so few positives, the named-pet classes are almost perfectly separable, and without a penalty the weights would grow without limit. We did not tune `C`: there is no data left to tune it on. The table below shows how sensitive the result is to it.

**A control with no learning.** A logistic regression with equal weights and no intercept is just the average of the members' logits, passed through the sigmoid. We score that as well. It separates two effects that stacking mixes together: averaging on the logit scale instead of the probability scale, and the weights the model actually learns.

In [3]:
def to_logits(probabilities: np.ndarray) -> np.ndarray:
    clipped = np.clip(probabilities, CLIP, 1 - CLIP)
    return np.log(clipped / (1 - clipped))


def features(members: np.ndarray, label_index: int, inputs: str) -> np.ndarray:
    '''Meta-model inputs for one label: its own three member logits, or all fifteen.'''
    logits = to_logits(members)
    if inputs == 'own label':
        return logits[:, :, label_index].T
    return logits.transpose(1, 0, 2).reshape(members.shape[1], -1)


def fit_stack(members: np.ndarray, targets: np.ndarray, c: float, inputs: str) -> list[LogisticRegression]:
    return [LogisticRegression(C=c, max_iter=1000).fit(features(members, index, inputs), targets[:, index]) for index in range(len(LABEL_COLUMNS))]


def predict_stack(models: list[LogisticRegression], members: np.ndarray, inputs: str) -> np.ndarray:
    return np.stack([model.predict_proba(features(members, index, inputs))[:, 1] for index, model in enumerate(models)], axis=1)


def logit_mean(members: np.ndarray) -> np.ndarray:
    return 1 / (1 + np.exp(-to_logits(members).mean(axis=0)))

## How we validate a two-level model

The meta-model cannot be scored on the fold-0 images it was fitted on: that would reward it for memorizing them. So we split fold 0 again. For each label, a stratified 5-fold split of the 2,452 fold-0 images fits the meta-model on four parts and predicts the fifth, until every image has an out-of-fold prediction. Macro F1 is computed from those predictions at 0.25. The members themselves never saw any fold-0 image, so this is a clean estimate for the second level.

With 8–10 positives, each inner part holds only about two photos of each pet, so one split can be lucky. We repeat the whole procedure 10 times with different shuffles and report the mean and the worst repeat. The two means have no parameters to fit, so they are scored once on all of fold 0.

Two limits remain. Exp3's and Exp5's checkpoints were themselves selected on fold 0, so all fold-0 scores here are optimistic. And on fold 0 one named-pet photo moves macro F1 by about 0.012, so the table can only show differences of whole photos.

In [4]:
def out_of_fold(c: float, inputs: str, repeat: int) -> np.ndarray:
    predictions = np.zeros(validation_targets.shape)
    for index in range(len(LABEL_COLUMNS)):
        x, y = features(validation_members, index, inputs), validation_targets[:, index]
        for train, held_out in StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=SEED + repeat).split(x, y):
            model = LogisticRegression(C=c, max_iter=1000).fit(x[train], y[train])
            predictions[held_out, index] = model.predict_proba(x[held_out])[:, 1]
    return predictions


def macro_f1(probabilities: np.ndarray, threshold: float = THRESHOLD) -> float:
    return score_probabilities(validation_targets, probabilities, threshold)['macro_f1']


fixed = {'probability mean (0.98047)': mean_probabilities(list(validation_members)), 'logit mean (no learning)': logit_mean(validation_members)}
header = f"{'combination':<36} {'F1@0.25':>8} {'worst':>8} " + ' '.join(f'{label:>8}' for label in LABEL_COLUMNS) + f" {'F1@0.5':>8}"
print(header)
print('-' * len(header))
for name, probabilities in fixed.items():
    per_label = score_probabilities(validation_targets, probabilities, THRESHOLD)['per_label_f1']
    print(f"{name:<36} {macro_f1(probabilities):>8.5f} {'':>8} " + ' '.join(f'{per_label[label]:>8.4f}' for label in LABEL_COLUMNS) + f' {macro_f1(probabilities, 0.5):>8.5f}')

stacked_oof = {}
for inputs in ('own label', 'all labels'):
    for c in C_GRID:
        repeats = [out_of_fold(c, inputs, repeat) for repeat in range(REPEATS)]
        stacked_oof[(inputs, c)] = repeats
        at_threshold = [score_probabilities(validation_targets, p, THRESHOLD) for p in repeats]
        per_label = {label: np.mean([s['per_label_f1'][label] for s in at_threshold]) for label in LABEL_COLUMNS}
        macro = [s['macro_f1'] for s in at_threshold]
        marker = '  <- used' if (inputs, c) == ('own label', C) else ''
        print(f"{f'LR, {inputs}, C={c:g}':<36} {np.mean(macro):>8.5f} {min(macro):>8.5f} " + ' '.join(f'{per_label[label]:>8.4f}' for label in LABEL_COLUMNS)
              + f' {np.mean([macro_f1(p, 0.5) for p in repeats]):>8.5f}{marker}')
print(f'\nLR rows: out-of-fold on fold 0, mean and worst of {REPEATS} repeats of {INNER_FOLDS}-fold CV.')

combination                           F1@0.25    worst      Cat      Dog    Moris    Motya  Biatrix   F1@0.5
------------------------------------------------------------------------------------------------------------
probability mean (0.98047)            0.98656            0.9956   0.9960   1.0000   1.0000   0.9412  0.99891
logit mean (no learning)              0.99901            0.9975   0.9975   1.0000   1.0000   1.0000  0.99911
LR, own label, C=0.01                 0.99897  0.99871   0.9971   0.9978   1.0000   1.0000   1.0000  0.98552
LR, own label, C=0.1                  0.98704  0.98685   0.9964   0.9977   1.0000   1.0000   0.9412  0.99896
LR, own label, C=1                    0.99878  0.99861   0.9963   0.9977   1.0000   1.0000   1.0000  0.99899  <- used
LR, own label, C=10                   0.99877  0.99861   0.9962   0.9977   1.0000   1.0000   1.0000  0.99896
LR, all labels, C=0.01                0.99893  0.99871   0.9969   0.9978   1.0000   1.0000   1.0000  0.98565
LR, all la

### Which fold-0 images make the difference

Every fold-0 image where the probability mean, the logit mean or the stacked model (first repeat, out-of-fold) makes a mistake, with each member's probability. For Cat and Dog only the counts are shown.

In [5]:
candidates = {'prob mean': fixed['probability mean (0.98047)'], 'logit mean': fixed['logit mean (no learning)'], 'stacked': stacked_oof[('own label', C)][0]}
decisions = {name: probabilities >= THRESHOLD for name, probabilities in candidates.items()}
truth = validation_targets.astype(bool)
for name, decided in decisions.items():
    print(f'{name:<11} errors per label (FP/FN):', {label: f'{(decided & ~truth)[:, i].sum()}/{(~decided & truth)[:, i].sum()}' for i, label in enumerate(LABEL_COLUMNS)})
print(f"\n{'image_id':<14} {'label':<8} {'true':>4} " + ' '.join(f'{name:>7}' for name in MEMBERS) + ' ' + ' '.join(f'{name:>11}' for name in candidates))
for label in ('Moris', 'Motya', 'Biatrix'):
    index = LABEL_COLUMNS.index(label)
    for row in np.flatnonzero(np.any([decided[:, index] != truth[:, index] for decided in decisions.values()], axis=0)):
        print(f'{validation_ids[row][:12]:<14} {label:<8} {int(truth[row, index]):>4} ' + ' '.join(f'{p:>7.3f}' for p in validation_members[:, row, index])
              + ' ' + ' '.join(f'{probabilities[row, index]:>11.3f}' for probabilities in candidates.values()))

prob mean   errors per label (FP/FN): {'Cat': '8/1', 'Dog': '7/1', 'Moris': '0/0', 'Motya': '0/0', 'Biatrix': '1/0'}
logit mean  errors per label (FP/FN): {'Cat': '4/1', 'Dog': '3/2', 'Moris': '0/0', 'Motya': '0/0', 'Biatrix': '0/0'}
stacked     errors per label (FP/FN): {'Cat': '5/2', 'Dog': '5/1', 'Moris': '0/0', 'Motya': '0/0', 'Biatrix': '0/0'}

image_id       label    true    exp1    exp3    exp5   prob mean  logit mean     stacked
520c9a44013a   Biatrix     0   0.642   0.174   0.054       0.290       0.218       0.244


## Fit on all of fold 0 and look at the weights

The final meta-model is fitted once on all 2,452 fold-0 images with `C = 1` and the own-label inputs. Its weights show what it learned. Equal weights would mean it only rescales the logit mean. A weight near zero would mean a member is ignored for that label.

In [6]:
models = fit_stack(validation_members, validation_targets, C, 'own label')
print(f"{'label':<8} " + ' '.join(f'{name:>7}' for name in MEMBERS) + f" {'intercept':>10}")
for label, model in zip(LABEL_COLUMNS, models):
    print(f'{label:<8} ' + ' '.join(f'{weight:>7.3f}' for weight in model.coef_[0]) + f' {model.intercept_[0]:>10.2f}')
print('\nfold-0 prevalence:', dict(zip(LABEL_COLUMNS, validation_targets.mean(axis=0).round(4).tolist())))

label       exp1    exp3    exp5  intercept
Cat        0.339   0.439  -0.041      -1.81
Dog        0.471   0.135   0.106      -0.38
Moris      0.351   0.392   0.354      -0.03
Motya      0.274   0.319   0.307      -0.04
Biatrix    0.137   0.335   0.468      -0.20

fold-0 prevalence: {'Cat': 0.4139, 'Dog': 0.4095, 'Moris': 0.0041, 'Motya': 0.0037, 'Biatrix': 0.0033}


## Test set: what does stacking change?

We apply the fitted meta-model to the members' saved test probabilities and compare its 0/1 decisions with the 0.98047 submission (the probability mean). The logit-mean column shows how much of the change needs no learning at all. On the test set one named-pet photo is worth about 0.0007 macro F1 and a Cat or Dog cell about 0.00001, so only the named-pet changes can move the score noticeably.

The intercepts were fitted to fold 0, where each pet is 0.3–0.4% of images. If the test set holds a larger share of named pets, a model calibrated to fold 0 will lean towards predicting too few of them.

In [7]:
test_candidates = {'prob mean': mean_probabilities(list(test_members)), 'logit mean': logit_mean(test_members), 'stacked': predict_stack(models, test_members, 'own label')}
test_decisions = {name: probabilities >= THRESHOLD for name, probabilities in test_candidates.items()}
best = read_probabilities(ROOT / BEST_SUBMISSION, test_ids).astype(bool)
if not np.array_equal(best, test_decisions['prob mean']):
    print('Note: the recomputed probability mean differs from the saved 0.98047 file in', int((best != test_decisions['prob mean']).sum()), 'cells.')
print(f"{'label':<8} " + ' '.join(f'{name:>10}' for name in test_candidates))
for index, label in enumerate(LABEL_COLUMNS):
    print(f'{label:<8} ' + ' '.join(f'{int(decided[:, index].sum()):>10}' for decided in test_decisions.values()))
print('predicted share of each label (prob mean):', dict(zip(LABEL_COLUMNS, test_decisions['prob mean'].mean(axis=0).round(4).tolist())))
for a, b in (('stacked', 'prob mean'), ('logit mean', 'prob mean'), ('stacked', 'logit mean')):
    changed = test_decisions[a] != test_decisions[b]
    print(f'{a} vs {b}: {int(changed.sum())} cells differ', {label: int(changed[:, i].sum()) for i, label in enumerate(LABEL_COLUMNS)})

label     prob mean logit mean    stacked
Cat            7833       7789       7781
Dog            7718       7664       7669
Moris           167        159        158
Motya           137        128        128
Biatrix         185        180        177
predicted share of each label (prob mean): {'Cat': 0.4195, 'Dog': 0.4133, 'Moris': 0.0089, 'Motya': 0.0073, 'Biatrix': 0.0099}
stacked vs prob mean: 127 cells differ {'Cat': 52, 'Dog': 49, 'Moris': 9, 'Motya': 9, 'Biatrix': 8}
logit mean vs prob mean: 120 cells differ {'Cat': 44, 'Dog': 54, 'Moris': 8, 'Motya': 9, 'Biatrix': 5}
stacked vs logit mean: 45 cells differ {'Cat': 18, 'Dog': 23, 'Moris': 1, 'Motya': 0, 'Biatrix': 3}


## Write the stacking submission

With `BUILD_SUBMISSION=True` (environment variable `STAGE2_BUILD_SUBMISSION=1`), this cell writes the stacked model's 0/1 test decisions at 0.25 to `submissions/`. It also writes the first repeat's out-of-fold fold-0 probabilities and a run record under `experiments/exp4/results/<run-id>/`. Nothing is overwritten. Record the public score in the run record and the README after a manual upload.

In [8]:
if BUILD_SUBMISSION:
    result_dir = ROOT / 'experiments' / 'exp4' / 'results' / RUN_ID
    submission_path = ROOT / 'submissions' / f"exp4_stacking_logreg_exp1_exp3_exp5_{round(THRESHOLD * 100):03d}.csv"
    git_commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=ROOT, capture_output=True, text=True).stdout.strip() or None
    source_files = ('notebooks/15_stacking_logreg.ipynb', 'stage2/ensemble.py', 'stage2/experiment.py')
    oof_repeats = stacked_oof[('own label', C)]
    write_submission(submission_path, form_submission(DATA_DIR / 'sample_submission.csv', test_ids, test_candidates['stacked'], THRESHOLD))
    write_probabilities(result_dir / f'{RUN_ID}_val.csv', validation_ids, oof_repeats[0], VALIDATION_FOLD)
    write_run_record(result_dir / f'{RUN_ID}.json', {
        'run_id': RUN_ID, 'experiment': '04_ensemble', 'status': 'completed', 'fold': VALIDATION_FOLD,
        'method': 'per-label logistic regression on the members\' own-label logits (L2, C=1), fitted on fold-0 predictions, then fixed threshold',
        'members': [{'experiment': name, 'run_id': run_id, 'checkpoint_sha256': records[name]['artifacts']['best_checkpoint_sha256']} for name, run_id in MEMBERS.items()],
        'meta_model': {label: {'weights': dict(zip(MEMBERS, map(float, model.coef_[0]))), 'intercept': float(model.intercept_[0])} for label, model in zip(LABEL_COLUMNS, models)},
        'validation': f'{REPEATS} repeats of stratified {INNER_FOLDS}-fold CV inside fold 0, per label; out-of-fold macro F1 at {THRESHOLD}',
        'data': {'fold_file': 'folds.csv', 'fold_sha256': fold_sha256, 'label_columns': list(LABEL_COLUMNS)},
        'metrics': {'threshold': THRESHOLD, 'macro_f1_oof_mean': float(np.mean([macro_f1(p) for p in oof_repeats])),
                    'macro_f1_oof_worst': float(min(macro_f1(p) for p in oof_repeats)),
                    'probability_mean_macro_f1': macro_f1(fixed['probability mean (0.98047)'])},
        'artifacts': {'validation_probabilities_csv': str((result_dir / f'{RUN_ID}_val.csv').relative_to(ROOT)), 'submission_csv': str(submission_path.relative_to(ROOT))},
        'git_commit': git_commit, 'source_sha256': {path: file_sha256(ROOT / path) for path in source_files},
        'public_leaderboard_score': None, 'error': None,
    })
    print('CSV ready for manual review:', submission_path.relative_to(ROOT))
    print('Run record:', (result_dir / f'{RUN_ID}.json').relative_to(ROOT))
else:
    print('No submission CSV built or uploaded (set STAGE2_BUILD_SUBMISSION=1).')

No submission CSV built or uploaded (set STAGE2_BUILD_SUBMISSION=1).


## Findings before submission

| Combination | Fold-0 macro F1 at 0.25 | Public leaderboard |
|---|---:|---:|
| Probability mean of Exp1, Exp3, Exp5 (notebook 13) | 0.98656 | **0.98047** |
| Logit mean, no learning (control) | 0.99901 | not submitted |
| **Logistic-regression stacking**, own-label logits, C = 1 (this notebook) | 0.99878 out-of-fold (worst repeat 0.99861) | to be filled after upload |

The fold-0 figure for stacking is out-of-fold, from 10 repeats of 5-fold CV inside fold 0. The two means have no parameters and are scored on all of fold 0.

**1. On fold 0, the whole gain is one photo.** Stacking and the logit mean both score about 0.999, against 0.987 for the probability mean. The only named-pet error that differs is one Biatrix false alarm (`520c9a44013a`): Exp1 gives it 0.64 and the other two 0.17 and 0.05. The probability mean lands at 0.29, just above 0.25. The logit mean (0.22) and the stacked model (0.24) land just below. One photo of eight Biatrix positives is worth 0.012 macro F1 here, so this is a single observation, not evidence of a better method. The sensitivity rows show the same thing: with `C = 0.1` the same photo goes back above 0.25 and the score falls to 0.987.

**2. The learned weights add little beyond averaging on the logit scale.** For the named pets, the three weights are close to equal (for example Moris 0.35, 0.39, 0.35) and the intercepts are close to 0. The model is then almost exactly the logit mean, slightly rescaled. On the test set, stacking and the logit mean differ in only 4 named-pet cells. Their changes against the 0.98047 file mostly coincide (26 and 22 named-pet cells). Only for Cat does the model learn something different: it nearly ignores Exp5 (weight −0.04), and Exp5's Cat predictions were its weakest.

**3. What changes on the test set: fewer named pets.** The stacked model removes 26 named-pet positives and adds none (Moris 167 → 158, Motya 137 → 128, Biatrix 185 → 177). The reason is the logit scale. The probability mean says "pet" when one member is very confident, because one 0.99 among three members is already a mean of 0.33. A mean of logits needs more agreement. The removed cells are therefore those backed by a single confident member. Notebook 13 showed that some single-member calls, BiT's Motya calls in particular, were what helped the three-model mean. Fold 0 has no example of such a case. The test set also appears to hold about twice the fold-0 share of named pets (0.7–1.0% predicted, against 0.3–0.4%), and an intercept fitted to fold 0 leans against them. We therefore expect the stacked file to score at or below 0.98047, but only the leaderboard can tell.

**4. More inputs overfit.** Giving each label all 15 member logits works only with strong regularization. With `C = 10` the worst repeat drops to 0.967, because the model fits the 8–10 pet photos of the training split. This is the expected behaviour of a second-level model with far more parameters than positive examples.

**Conclusion.** Stacking is valid here: the meta-model is fitted only on held-out member predictions and validated by cross-validation inside fold 0. But fold 0 is too small for it to learn more than "average the logits". Its local gain rests on one photo. We keep the equal-weight probability mean as the main submission and upload `exp4_stacking_logreg_exp1_exp3_exp5_025.csv` to compare the leaderboard with this local result, as the assignment asks. Learning the weights reliably would need out-of-fold predictions for all five folds (about 125 named-pet photos instead of 27).